# Votaciones nominales de la Cámara de Diputados — LXVI Legislatura (2024-2027)

Este notebook descarga de la [Gaceta Parlamentaria](https://gaceta.diputados.gob.mx/gp_votaciones.html) cómo votó cada diputado en cada votación de la LXVI Legislatura. El resultado es una base de datos en formato *largo*, con **una fila por diputado por votación**.

| columna | contenido |
|---|---|
| `anio`, `periodo` | año legislativo y periodo de sesiones |
| `fecha`, `fecha_iso` | fecha de la sesión (texto original y `AAAA-MM-DD`) |
| `asunto` | decreto, dictamen o acuerdo que se vota |
| `descripcion_votacion` | qué votación es (p. ej. *en lo general* o *en lo particular*) |
| `url_tabla` | enlace a la tabla oficial de la votación (identificador único) |
| `diputado`, `partido` | nombre del diputado y su grupo parlamentario |
| `sentido` | `A favor`, `En contra`, `Abstención`, `Quórum` o `Ausente` |

### ¿Cómo está organizado el sitio?

La información está repartida en cuatro niveles de páginas:

1. **Índice** (`gp_votaciones.html`): enlaces a una página por periodo de sesiones.
2. **Página del periodo** (p. ej. `vot66_a1primero.html`): cada fecha es un `<font color="#CC0000">` y abajo vienen los asuntos votados ese día, cada uno en un `<li>`. Un asunto puede tener **varias votaciones** (en lo general y en lo particular), cada una con un enlace *Votación*.
3. **Tabla de la votación** (`tabla1or1-2.php3`): conteo por sentido del voto y por partido. Los números son **botones de un formulario**.
4. **Lista de diputados**: se obtiene enviando ese formulario (una petición POST). El botón de la columna *Total* devuelve a todos los diputados con ese sentido, agrupados por partido.

## Paso 0. Librerías y configuración

- `requests`: descarga las páginas (GET) y envía los formularios (POST).
- `BeautifulSoup`: lee el HTML.
- `pandas`: arma la tabla final y la guarda en CSV.

Usamos una `Session` para reutilizar la conexión con el servidor y ponemos una pausa entre peticiones para no saturarlo.

In [ ]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = "https://gaceta.diputados.gob.mx"
INDICE = BASE + "/gp_votaciones.html"
PAUSA = 0.5       # segundos entre peticiones
REINTENTOS = 3    # intentos por petición si falla la red

session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (proyecto de investigacion academica)"

## Paso 1. Funciones auxiliares

- `get_soup(url, data=None)`: descarga una página y la devuelve lista para buscar en ella. Si recibe `data`, envía un POST (así se "oprime" un botón del formulario). Si la petición falla, espera y reintenta. Las páginas de la Gaceta vienen en codificación Latin-1, así que la indicamos para que los acentos y la ñ salgan bien.
- `limpiar(texto)`: junta espacios y saltos de línea repetidos en un solo espacio.
- `fecha_iso(fecha)`: convierte `"Martes 3 de septiembre de 2024"` a `"2024-09-03"`, un formato que se puede ordenar y filtrar.

In [ ]:
MESES = {m: i for i, m in enumerate(
    ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
     "septiembre", "octubre", "noviembre", "diciembre"], 1)}


def get_soup(url, data=None):
    for intento in range(1, REINTENTOS + 1):
        time.sleep(PAUSA)
        try:
            resp = session.post(url, data=data, timeout=30) if data else session.get(url, timeout=30)
            resp.raise_for_status()
            break
        except requests.RequestException:
            if intento == REINTENTOS:
                raise
            time.sleep(5 * intento)
    resp.encoding = "cp1252"  # las páginas de la Gaceta vienen en Latin-1
    return BeautifulSoup(resp.text, "html.parser")


def limpiar(texto):
    return re.sub(r"\s+", " ", texto).strip()


def fecha_iso(fecha):
    m = re.search(r"(\d{1,2}) de (\w+) de (\d{4})", fecha or "")
    if not m or m.group(2).lower() not in MESES:
        return None
    return f"{m.group(3)}-{MESES[m.group(2).lower()]:02d}-{int(m.group(1)):02d}"


fecha_iso("Martes 3 de septiembre de 2024")

## Paso 2. Listar los periodos de la legislatura

En el índice, los enlaces de la LXVI Legislatura tienen la forma `/Gaceta/Votaciones/66/vot66_...html`. El nombre del archivo indica el año legislativo: `a1` es el primer año, `a2` el segundo y `a3` el tercero. El nombre del periodo y sus meses se sacan de la celda donde está el enlace.

Buscar los enlaces por su patrón funciona igual que el XPath `/html/body/table[6]/.../td[2]/table[1]`, pero no se rompe si el sitio agrega o mueve una tabla.

In [ ]:
ANIOS = {"1": "Primer año", "2": "Segundo año", "3": "Tercer año"}


def listar_periodos(legislatura="66"):
    soup = get_soup(INDICE)
    periodos = []
    for a in soup.find_all("a", href=re.compile(rf"/Votaciones/{legislatura}/vot{legislatura}_")):
        anio = re.search(r"_a(\d)", a["href"])
        periodos.append({
            "anio": ANIOS.get(anio.group(1)) if anio else None,
            "periodo": limpiar(a.find_parent("td").get_text()),
            "url_periodo": urljoin(BASE, a["href"]),
        })
    return periodos


periodos = listar_periodos("66")
pd.DataFrame(periodos)

## Paso 3. Leer una página de periodo: fechas, asuntos y enlaces a las votaciones

Recorremos en orden todos los `<font>` y `<li>` de la página:

- Un `<font color="#CC0000">` es una **fecha**. La guardamos y se la asignamos a todos los `<li>` que vienen después, hasta la siguiente fecha. Esto equivale a recorrer `body > font:nth-child(n)`.
- Un `<li>` es un **asunto votado**. Su texto (el decreto) es lo que aparece antes del primer `<br>`.
- Dentro del `<li>`, cada enlace *Votación* es una votación distinta. Para cada uno guardamos el renglón de texto que lo acompaña (lo que va desde el `<br>` anterior hasta el enlace), que dice si es la votación en lo general o en lo particular.

In [ ]:
def texto_de(nodo):
    return nodo.get_text() if hasattr(nodo, "get_text") else str(nodo)


def parse_periodo(url):
    soup = get_soup(url)
    votaciones = []
    fecha = None
    for el in soup.body.find_all(["font", "li"]):
        # Una fecha: se aplica a los <li> que siguen
        if el.name == "font" and el.get("color", "").lower() == "#cc0000":
            fecha = limpiar(el.get_text())
            continue
        if el.name != "li":
            continue

        # Texto del asunto: todo lo que está antes del primer <br>
        partes = []
        for nodo in el.children:
            if getattr(nodo, "name", None) == "br":
                break
            partes.append(texto_de(nodo))
        asunto = limpiar("".join(partes))

        # Cada enlace "Votación" con su renglón descriptivo
        for a in el.find_all("a", string=re.compile(r"Votaci")):
            renglon = []
            for nodo in a.previous_siblings:
                if getattr(nodo, "name", None) == "br":
                    break
                renglon.append(texto_de(nodo))
            votaciones.append({
                "fecha": fecha,
                "fecha_iso": fecha_iso(fecha),
                "asunto": asunto,
                "descripcion_votacion": limpiar("".join(reversed(renglon))),
                "url_tabla": urljoin(BASE, a["href"]),
            })
    return votaciones


# Prueba con el primer periodo
ejemplo = parse_periodo(periodos[0]["url_periodo"])
print(len(ejemplo), "votaciones")
pd.DataFrame(ejemplo).head()

## Paso 4. Leer la tabla de una votación

La tabla `.php3` es un formulario (`<form>`). De ella necesitamos:

- **`action`**: la dirección a la que se envía el formulario. Cambia según el periodo, por ejemplo `/voto66/ordi11/lanordi11.php3`.
- **Campos ocultos** (`evento`, `nomtit`): identifican la votación y se envían junto con el botón.
- **Botones de la columna *Total***: uno por renglón (Favor, Contra, Abstención, Quórum, Ausente). Se llaman `lola[1Y]`, donde `Y` es el sentido del voto, y su valor es el número de diputados. Cuando el conteo es 0 no hay botón (solo un `input type="button"`), así que ese sentido se omite.

*Quórum* significa que el diputado pasó lista de asistencia pero no votó.

In [ ]:
SENTIDOS = {
    "Favor": "A favor",
    "Contra": "En contra",
    "Abstención": "Abstención",
    "Quórum *": "Quórum",
    "Ausente": "Ausente",
}


def parse_tabla(url):
    soup = get_soup(url)
    form = soup.find("form")
    action = urljoin(url, form["action"])
    ocultos = {i["name"]: i.get("value", "") for i in form.find_all("input", type="hidden")}

    botones = {}  # sentido -> (name, value) del botón de la columna Total
    for tr in form.find_all("tr"):
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 2:
            continue
        etiqueta = limpiar(tds[0].get_text())
        if etiqueta not in SENTIDOS:
            continue
        boton = tds[1].find("input", type="submit")
        if boton:
            botones[SENTIDOS[etiqueta]] = (boton["name"], boton["value"])
    return action, ocultos, botones


action, ocultos, botones = parse_tabla(ejemplo[1]["url_tabla"])
print(action)
botones

## Paso 5. Obtener la lista de diputados (POST del formulario)

Enviar el formulario con un botón (`{"lola[11]": "359", "evento": ...}`) devuelve una página con un bloque por partido:

> Diputados de **Morena** que votaron a favor: 250
> 1:Nombre Apellido<br>2:...

De cada encabezado sacamos el nombre del partido con una expresión regular, y de la tabla que le sigue, los nombres (las líneas con forma `número:nombre`).

In [ ]:
def parse_lista(action, ocultos, boton):
    soup = get_soup(action, data={**ocultos, boton[0]: boton[1]})
    diputados = []
    for center in soup.find_all("center"):
        m = re.match(r"Diputados (?:de la |del |de )?(.+?) que .*:\s*\d+", limpiar(center.get_text()))
        if not m:
            continue
        partido = m.group(1)
        partido = "Sin partido" if partido == "sin partido" else partido
        tabla = center.find_next("table")
        for td in tabla.find_all("td"):
            for linea in td.get_text("\n").split("\n"):
                nm = re.match(r"\s*\d+:\s*(.+)", linea)
                if nm:
                    diputados.append((partido, limpiar(nm.group(1))))
    return diputados


lista = parse_lista(action, ocultos, botones["En contra"])
print(len(lista), "diputados en contra")
lista[:5]

## Paso 6. Juntar todo

- `scrape_periodo(url)`: para cada votación del periodo lee la tabla y pide la lista de cada sentido. Además **comprueba** que el número de nombres leídos coincida con el conteo oficial del botón; si no coincide, imprime un aviso.
- `scrape_legislatura()`: hace lo mismo para todos los periodos y agrega las columnas `anio` y `periodo`.

In [ ]:
def scrape_periodo(url):
    votaciones = parse_periodo(url)
    print(f"{len(votaciones)} votaciones encontradas en {url}")
    filas = []
    for i, v in enumerate(votaciones, 1):
        print(f"[{i}/{len(votaciones)}] {v['fecha']} - {v['asunto'][:70]}")
        action, ocultos, botones = parse_tabla(v["url_tabla"])
        for sentido, boton in botones.items():
            lista = parse_lista(action, ocultos, boton)
            if len(lista) != int(boton[1]):
                print(f"   ! {sentido}: se esperaban {boton[1]} diputados, se leyeron {len(lista)}")
            for partido, nombre in lista:
                filas.append({**v, "diputado": nombre, "partido": partido, "sentido": sentido})
    return pd.DataFrame(filas)


def scrape_legislatura(legislatura="66"):
    dfs = []
    for p in listar_periodos(legislatura):
        print(f"\n=== {p['anio']} - {p['periodo']} ===")
        df = scrape_periodo(p["url_periodo"])
        dfs.append(df.assign(anio=p["anio"], periodo=p["periodo"]))
    df = pd.concat(dfs, ignore_index=True)
    columnas = ["anio", "periodo", "fecha", "fecha_iso", "asunto", "descripcion_votacion",
                "url_tabla", "diputado", "partido", "sentido"]
    return df[columnas]

## Paso 7. Prueba rápida con un solo periodo

Antes de la corrida completa, probamos con un periodo corto: el primer extraordinario del segundo año, con 5 votaciones (menos de un minuto).

In [ ]:
prueba = scrape_periodo(BASE + "/Gaceta/Votaciones/66/vot66_a2extra1.html")
prueba.head()

## Paso 8. Corrida completa y guardado

Recorre los 7 periodos (unas 300 votaciones y 1,200 peticiones). Tarda **10 a 15 minutos**.

El CSV se guarda en `utf-8-sig` para que Excel muestre bien los acentos.

In [ ]:
df = scrape_legislatura("66")
df.to_csv("votaciones_LXVI.csv", index=False, encoding="utf-8-sig")
print(len(df), "registros guardados")

## Paso 9. Validación

Revisiones básicas de calidad:

- Cuántos registros hay por sentido del voto y por partido.
- Si algún diputado aparece **dos veces en la misma votación** (debería ser 0).
- Las votaciones con menos de 500 registros. Esto **no es un error**: la tabla oficial reporta menos de 500 cuando hay curules vacantes o licencias.

In [ ]:
# Para retomar sin volver a descargar: df = pd.read_csv("votaciones_LXVI.csv")
print(df.sentido.value_counts(), "\n")
print(df.partido.value_counts(), "\n")
print("Votaciones:", df.url_tabla.nunique())
print("Diputados distintos:", df.diputado.nunique())
print("Duplicados dentro de una votación:", df.duplicated(["url_tabla", "diputado"]).sum())

por_votacion = df.groupby("url_tabla").size()
print("Votaciones con menos de 500 registros:", (por_votacion < 500).sum())

### Ejemplo: votos por partido en una votación

Una tabla cruzada de partido × sentido del voto para la primera votación de la reforma judicial.

In [ ]:
reforma = df[df.url_tabla.str.endswith("tabla1or1-2.php3")]
print(reforma.asunto.iloc[0], "\n")
pd.crosstab(reforma.partido, reforma.sentido, margins=True)